In [1]:
import tensorflow as tf
from keras.layers import*
from keras.models import Model
from keras.datasets import imdb
from requests import post
from torch.onnx.ops import attention

## Load the dataset

In [2]:
vocab_size = 20000
(x_train,y_train),(x_test,y_test) = imdb.load_data(num_words=vocab_size)

In [3]:
x_train.shape

(25000,)

In [4]:
y_test.shape

(25000,)

## Define hyperparameters

In [5]:
maxlen=200
embed_dimension=32
num_head=2
ff_dim=32


## Data Preprocessing

In [6]:
from keras.preprocessing.sequence import pad_sequences

In [7]:
x_train

array([list([1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]),
       list([1, 194, 1153, 194, 82

In [8]:
x_train = pad_sequences(x_train,maxlen = maxlen)
x_test =  pad_sequences(x_test,maxlen= maxlen)
x_train.shape

(25000, 200)

Build the model

In [9]:
# Input layer

inputs = Input(shape=(maxlen,))

# Token Embedding Layer
token_emb_layer = Embedding(input_dim=vocab_size, output_dim=embed_dimension )
x = token_emb_layer(inputs)

#position embedding layer
positions = tf.range(start=0, limit = maxlen,delta=1)
pos_emb_layer = Embedding(input_dim=maxlen, output_dim=embed_dimension)
position_emb = pos_emb_layer(positions)

# Add the token +position embedding

x = x + position_emb

#Add transformer block
#1.Multi-head self Attention
attention_output = MultiHeadAttention(num_heads=num_head, key_dim=embed_dimension)(x,x)
attention_output = Dropout(0.2)(attention_output)

#Residual connections(add + norm)
x1=LayerNormalization()(attention_output+x)

# feed-forward neural network
ffn = Dense(ff_dim, activation="relu")(x1)
ffn = Dense(embed_dimension)(ffn)
ffn = Dropout(0.1)(ffn)

#Residual connections(add + norm)
x2 = LayerNormalization()(x1+ffn)

#Classification Head

x3 = GlobalAveragePooling1D()(x2)
x3 = Dropout(0.1)(x3)
x3 = Dense(20,activation='relu')(x3)
x3 = Dropout(0.1)(x3)

##Output Layer
outputs = Dense(1, activation='sigmoid')(x3)


In [10]:
# Create object of model

model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 32)   │    640,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 200, 32)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 200, 32)   │      8,416 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 200, 32)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 200, 32)   │          0 │ dropout_1[0][0],  │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 200, 32)   │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 200, 32)   │      1,056 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 200, 32)   │      1,056 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 200, 32)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 200, 32)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 200, 32)   │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 20)        │        660 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 20)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         21 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 651,337 (2.48 MB)

 Trainable params: 651,337 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
from keras.utils import plot_model

In [12]:
plot_model(model,show_layer_names = True, show_layer_activations=True,show_shapes =True)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [13]:
# Compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


In [14]:
history = model.fit(x_train, y_train, batch_size=32,epochs=5, validation_data=(x_test, y_test)) 


Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 34s 40ms/step - accuracy: 0.8247 - loss: 0.3792 - val_accuracy: 0.8677 - val_loss: 0.3018
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.9271 - loss: 0.1938 - val_accuracy: 0.8666 - val_loss: 0.3273
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.9601 - loss: 0.1196 - val_accuracy: 0.8508 - val_loss: 0.4192
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.9766 - loss: 0.0771 - val_accuracy: 0.8469 - val_loss: 0.5583
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 39ms/step - accuracy: 0.9852 - loss: 0.0484 - val_accuracy: 0.8438 - val_loss: 0.5997


In [15]:
import numpy as np

In [16]:
new = x_test[100]

In [17]:
new = np.reshape(new, (1,maxlen))


In [18]:
new.shape

(1, 200)

In [19]:
model.predict(new)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step


array([[0.02026966]], dtype=float32)

## Prediction on the Trained Model

Encoding

### Evaluate the model on the full test set

In [28]:
# Evaluate overall performance of the trained model on the held-out test set
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8438 - loss: 0.5997
Test Loss: 0.5997
Test Accuracy: 0.8438


### Generate predictions for the full test set

In [29]:
# Predict probabilities for every review in the test set
y_pred_probs = model.predict(x_test)

# Convert probabilities to binary class labels using a 0.5 threshold
y_pred_labels = (y_pred_probs > 0.5).astype("int32").flatten()

print("Predicted probabilities (first 10):", y_pred_probs[:10].flatten())
print("Predicted labels (first 10):        ", y_pred_labels[:10])
print("Actual labels (first 10):           ", y_test[:10])

782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step
Predicted probabilities (first 10): [9.5009024e-04 9.9949706e-01 9.9866223e-01 4.4402465e-01 9.9382281e-01
 9.9925894e-01 9.9955302e-01 1.6466326e-03 9.9800712e-01 9.9967247e-01]
Predicted labels (first 10):         [0 1 1 0 1 1 1 0 1 1]
Actual labels (first 10):            [0 1 1 0 1 1 1 0 0 1]


### Classification report and confusion matrix

In [30]:
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_labels))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_labels, target_names=["Negative", "Positive"]))

Confusion Matrix:
[[10554  1946]
 [ 1959 10541]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.84      0.84      0.84     12500
    Positive       0.84      0.84      0.84     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000



### Predict on a single sample with a readable label

In [31]:
# Take a single test sample and predict its sentiment
sample = x_test[100]
sample = np.reshape(sample, (1, maxlen))

pred_prob = model.predict(sample)[0][0]
pred_label = "Positive" if pred_prob > 0.5 else "Negative"
actual_label = "Positive" if y_test[100] == 1 else "Negative"

print(f"Predicted probability: {pred_prob:.4f}")
print(f"Predicted sentiment:   {pred_label}")
print(f"Actual sentiment:      {actual_label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Predicted probability: 0.0203
Predicted sentiment:   Negative
Actual sentiment:      Positive


### Predict sentiment for custom, unseen text

In [35]:
word_index = imdb.get_word_index()
reversed_word_index = {value + 3: key for key, value in word_index.items()}
reversed_word_index[0] = '<PAD>'
reversed_word_index[1] = '<START>'
reversed_word_index[2] = '<UNK>'
reversed_word_index[3] = '<UNUSED>'


def decode_review(encoded_review):
    return " ".join(reversed_word_index.get(i, '?') for i in encoded_review if i != 0)
print(decode_review(x_test[100]))

<START> a quick glance at the premise of this film would seem to indicate just another dumb '80's inbred <UNK> slash fest the type where sex equals death and the actors are all annoying stereotypes you actually want to die however delivers considerably more br br rather than focus on bare flesh and gore though there is a little of each no sex however the flick focuses on delivering impending dread mounting tension amidst a lovely scenic backdrop these feelings are further heightened by a cast of realistically likable characters and antagonists that are more amoral than cardboard <UNK> of evil oh yeah george kennedy is here too and when is that not a good thing br br if you liked wrong turn then watch this to see where much of its' <UNK> came from


In [49]:
# Predict sentiment for your own review text using the trained model

def predict_sentiment(text, word_index=word_index, maxlen=maxlen):
    tokens = text.lower().split()
    encoded = [1] #<Start> token

    for word in tokens:
        idx = word_index.get(word, 2 - 3) + 3
        encoded.append(idx if idx < vocab_size else 2)

    padded = pad_sequences([encoded], maxlen=maxlen)
    prob = model.predict(padded, verbose=0)[0][0]
    label = "Positive" if prob > 0.5 else "Negative"
    return label, prob

review_text = "I just didn’t hate it much."
label, prob = predict_sentiment(review_text)
print(f"Review: {review_text}")
print(f"Predicted sentiment: {label} (probability: {prob:.4f})")

Review: I just didn’t hate it much.
Predicted sentiment: Negative (probability: 0.0201)
